<a href="https://colab.research.google.com/github/Luizadraeger/URBAN-DATA-ANALYTICS/blob/main/compute_server_buildingheight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Compute Server
Este notebook expõe uma função Python via API REST usando Flask + ngrok.

**Fluxo:**
1. Instala dependências
2. Define a função de cálculo (`compute`)
3. Sobe um servidor Flask na porta 5000
4. Cria um túnel público via ngrok
5. Exibe a URL pública para uso no cliente Python

> **Adaptação para Grasshopper/Rhino:** substitua o corpo de `compute()` pela lógica que você precisar (ex: análise GEE, cálculos geométricos, etc.)

In [1]:
#Célula 1: Instalar dependências
!pip install flask pyngrok --quiet
!pip install ghhops_server --quiet
!pip install rhino3dm --quiet
!pip install geemap --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.5 MB/s eta 0:00:00


In [3]:
%cd /content
!git clone -b pipeline https://github.com/Luizadraeger/URBAN-DATA-ANALYTICS.git
# Carrega a função buildfooprint para a memória do Python
%run /content/URBAN-DATA-ANALYTICS/pipeline-buildingheight.ipynb

/content
Cloning into 'URBAN-DATA-ANALYTICS'...
remote: Enumerating objects: 35, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 35 (delta 16), reused 6 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (35/35), 41.00 KiB | 4.55 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Mounted at /content/drive


In [2]:
#Célula 2: Autenticar ngrok
# Crie uma conta gratuita em https://ngrok.com e cole seu authtoken abaixo.
# Sem o token, o ngrok ainda funciona mas com limitações de conexão.

from pyngrok import ngrok

NGROK_AUTH_TOKEN = "2w8SB4S06qhuyxyKRWHRXoLs8R0_4XeByTV72ra1u9kECwHYE"  # ← cole seu token aqui (ou deixe vazio para modo free)

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✓ ngrok autenticado")
else:
    print("⚠  Rodando sem authtoken — funciona, mas pode haver limite de requisições")

✓ ngrok autenticado


In [4]:
from flask import Flask, request, jsonify
from pyngrok import ngrok
import traceback

app = Flask(__name__)
NGROK_AUTH_TOKEN = "2w8SB4S06qhuyxyKRWHRXoLs8R0_4XeByTV72ra1u9kECwHYE"

@app.route('/compute_footprints', methods=['POST'])
def compute_footprints():
    try:
        # Recebe lat, lon e radius do Grasshopper via Túnel
        data = request.get_json(force=True)
        lat = float(data['lat'])
        lon = float(data['lon'])
        radius = float(data.get('radius', 500))

        # Chama a função que foi importada do arquivo de pipeline
        buildings = buildfooprint(lat, lon, radius)

        # Retorna os metadados como string e os dados dos edifícios
        return jsonify({
            "status": "success",
            "lat": str(lat),
            "lon": str(lon),
            "radius": str(radius),
            "data": buildings
        })
    except Exception as e:
        print(traceback.format_exc())
        return jsonify({"status": "error", "message": str(e)}), 500

# --- PASSO 3: Lançamento ---
if __name__ == '__main__':
    ngrok.kill()
    if NGROK_AUTH_TOKEN:
        ngrok.set_auth_token(NGROK_AUTH_TOKEN)

    public_url = ngrok.connect(5000).public_url
    print(f"\n🚀 SERVIDOR ATIVO\nURL PARA O GRASSHOPPER: {public_url}/compute_footprints\n")

    app.run(port=5000, debug=False, use_reloader=False)


🚀 SERVIDOR ATIVO
URL PARA O GRASSHOPPER: https://e2cf-34-75-65-231.ngrok-free.app/compute_footprints

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [09/May/2026 23:23:35] "POST /compute_footprints HTTP/1.1" 200 -


### Observação:
Se, após executar o código acima, você ainda encontrar problemas com o limite de sessões do ngrok, pode ser que existam processos ngrok rodando em segundo plano de sessões anteriores do Colab que não foram encerradas corretamente. Nesses casos, a solução mais garantida é reiniciar o ambiente de execução do Colab (vá em `Runtime` > `Restart runtime` no menu superior).

In [ ]:
from pyngrok import ngrok

print("Encerrando quaisquer processos ngrok ativos...")
ngrok.kill()
print("Processos ngrok encerrados.")

Encerrando quaisquer processos ngrok ativos...
Processos ngrok encerrados.
